# 🦟 Malaria Cell Classification With PyTorch

## Medical AI Project For AltayBioAI

This Project Demonstrates A Lightweight Convolutional Neural Network (CNN) For Malaria Cell Classification Using PyTorch.

The Goal Is To Classify Microscopic Blood Cell Images Into Two Categories:

- Parasitized
- Uninfected

The Project Is Designed To Be Lightweight, Educational, And Easy To Reproduce On Consumer Hardware.

## 📚 Dataset

Dataset: Cell Images For Detecting Malaria

Classes:

- Parasitized
- Uninfected

## 🧠 Required Libraries

Install The Required Libraries Before Running The Notebook.

# Optional Installation

# pip install kagglehub
# pip install torch torchvision matplotlib

In [ ]:
import kagglehub
import os
import torch
import torchvision
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [ ]:
path = kagglehub.dataset_download(
    "iarunava/cell-images-for-detecting-malaria"
)

print(path)

In [ ]:
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])

In [ ]:
dataset_path = os.path.join(
    path,
    "cell_images"
)

dataset = ImageFolder(
    dataset_path,
    transform=transform
)

In [ ]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    dataset,
    [train_size, val_size]
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32
)

In [ ]:
class MalariaCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)

        return x

In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

num_classes = len(dataset.classes)

model = MalariaCNN(
    num_classes=num_classes
).to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

## ⚡ Training Note
The Default Number Of Epochs Is Kept Low To Reduce Training Time.
Increasing The Number Of Epochs May Improve Accuracy.

In [ ]:
epochs = 2

for epoch in range(epochs):
    model.train()
    running_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(
        f"Epoch {epoch+1}: "
        f"{running_loss:.4f}"
    )

In [ ]:
torch.save(
    model.state_dict(),
    "malaria_cnn.pth"
)

print("Model Saved Successfully!")

In [ ]:
correct = 0
total = 0

model.eval()

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f"Accuracy: {accuracy:.2f}%")

In [ ]:
images, labels = next(iter(val_loader))
images = images.to(device)
model.eval()

with torch.no_grad():
    outputs = model(images)
    _, preds = torch.max(outputs, 1)

plt.figure(figsize=(12,6))

for i in range(6):
    plt.subplot(2,3,i+1)
    img = images[i].cpu().permute(1,2,0)
    plt.imshow(img)
    plt.title(
        f"P:{preds[i].item()} T:{labels[i].item()}"
    )

    plt.axis("off")

plt.show()